In [56]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.cluster._kmeans")
import random
from matplotlib.colors import ListedColormap
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import norm
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.ndimage import convolve
import pickle

# [Zhou et al 2020] spatial aided GMM SAGMM

In [2]:
class SpatiallyAidedGMM:
    def __init__(self, n_clusters=2, n_spatial_components=2, max_iter=100):
        self.n_clusters = n_clusters
        self.n_spatial_components = n_spatial_components
        self.max_iter = max_iter
        self.gmm_feature = GaussianMixture(n_components=n_clusters, max_iter=max_iter)
        self.gmm_spatial = [GaussianMixture(n_components=n_spatial_components, max_iter=max_iter) for _ in range(n_clusters)]

    def fit(self, X, S):
        """
        X: Feature matrix (kinematic data) of shape (M, D)
        S: Spatial coordinates of shape (M, 2)
        """
        # Step 1: Fit GMM on kinematic features
        self.gmm_feature.fit(X)
        cluster_assignments = self.gmm_feature.predict(X)
        
        # Step 2: Fit spatial GMM for each cluster
        for k in range(self.n_clusters):
            cluster_indices = np.where(cluster_assignments == k)[0]
            if len(cluster_indices) > self.n_spatial_components:
                self.gmm_spatial[k].fit(S[cluster_indices])
            else:
                self.gmm_spatial[k] = None  # Avoid fitting on very few points
        
    def predict_proba(self, X, S):
        """
        Compute the probability of each point belonging to a cluster.
        """
        feature_probs = self.gmm_feature.predict_proba(X)
        spatial_probs = np.zeros_like(feature_probs)
        
        for k in range(self.n_clusters):
            if self.gmm_spatial[k] is not None:
                spatial_probs[:, k] = np.exp(self.gmm_spatial[k].score_samples(S))
        
        # Normalize spatial probabilities
        spatial_probs /= spatial_probs.sum(axis=1, keepdims=True) + 1e-10
        
        # Compute final probability as the product of feature and spatial probabilities
        final_probs = feature_probs * spatial_probs
        final_probs /= final_probs.sum(axis=1, keepdims=True) + 1e-10
        return final_probs
    
    def predict(self, X, S):
        """
        Assign each point to the most probable cluster.
        """
        return np.argmax(self.predict_proba(X, S), axis=1)



# [Zao et al 2016] Spatial GMM SGMM

In [32]:
def extract_sliding_windows(image, window_size=3):
    pad = window_size // 2
    padded_image = np.pad(image, pad, mode='edge')
    windows = []
    centers = []
    for i in range(pad, pad + image.shape[0]):
        for j in range(pad, pad + image.shape[1]):
            window = padded_image[i-pad:i+pad+1, j-pad:j+pad+1].flatten()
            windows.append(window)
            centers.append((i-pad, j-pad))
    return np.array(windows), centers
def sgmm(image, K=3, window_size=3, max_iter=10, tol=1e-4,random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    windows, centers = extract_sliding_windows(image, window_size)
    M, N = windows.shape
    D = 1  # grayscale


    flat_pixels = image.reshape(-1, 1)
    kmeans = KMeans(n_clusters=K, n_init=10, random_state=random_state).fit(flat_pixels)
    
    # Initial mean µ_k from kmeans centers
    mu = kmeans.cluster_centers_.flatten()
    
    
    sigma2 = np.zeros((K, 1))
    #labels = kmeans.labels_.reshape(image.shape)
    for k in range(K):
        pixels_in_k = flat_pixels[kmeans.labels_ == k]
        sigma2[k, 0] = np.var(pixels_in_k) + 1e-4 
    alpha = np.ones((M, K)) / K
    rng = np.random.RandomState(random_state)
    tau = rng.dirichlet(np.ones(K), (M, N))
    L_prev = None
    grad_L = 1
    t = 0
    
    while grad_L > tol and t < max_iter:
        # --- M-step ---
        for k in range(K):
            numerator = np.sum(tau[:, :, k] * windows)
            denominator = np.sum(tau[:, :, k])
            mu[k] = numerator / (denominator + 1e-12)

            sigma2[k, 0] = np.sum(tau[:, :, k] * (windows - mu[k])**2) / (denominator + 1e-12)

        alpha = np.sum(tau, axis=1) / N

        # --- E-step ---
        pdfs = np.zeros((M, N, K))
        for k in range(K):
            pdfs[:, :, k] = norm.pdf(windows, mu[k], np.sqrt(sigma2[k, 0]))
        weighted_pdfs = alpha[:, np.newaxis, :] * pdfs
        denominator = np.sum(weighted_pdfs, axis=2, keepdims=True) + 1e-12
        tau = weighted_pdfs / denominator

        # --- Log-likelihood ---
        L_new = 0
        L_new = np.sum(np.log(np.sum(weighted_pdfs, axis=2) + 1e-12)) / N
    

        # --- Verbosity ---
        if L_prev is not None:
            grad_L = np.abs(L_prev - L_new) / (np.abs(L_prev) + 1e-12)
        else:
            grad_L = np.inf

        # print(f"Iter {t:02d} | LogL = {L_new:.5f} | ∇L = {grad_L:.5e}",end="/")
        # print(f"   μ: {mu}")
        # print(f"   σ²: {sigma2.flatten()}")

        L_prev = L_new
        label_image = np.zeros_like(image, dtype=int)
        labels = np.argmax(alpha, axis=1)
        for idx, (i, j) in enumerate(centers):
            label_image[i, j] = labels[idx]
        # plt.imshow(label_image,cmap='gray')
        # plt.show()
        t += 1

    return label_image, mu, sigma2,alpha

# [Wang 2012, GMM+HMRF]

In [33]:
def gmm_hmrf_from_gmm(
    X,
    gmm,
    image_shape,
    beta=1.0,
    n_gmm_components=1,
    max_em_iter=10,
    max_icm_iter=5,
    tol=1e-3,
    random_state=0,
):
    """
    GMM-HMRF segmentation using a pre-fitted GMM.

    Parameters
    ----------
    X : np.ndarray (N, d)
    gmm : GaussianMixture
        Pre-fitted global GMM
    image_shape : (H, W)
    beta : float
    n_gmm_components : int
    """

    rng = np.random.RandomState(random_state)

    H, W = image_shape
    n_labels = gmm.n_components
    d = X.shape[1]

    # --- Initialization from GMM
    labels = gmm.predict(X).reshape(H, W)

    pixels = X.copy()

    # --- Initialize one GMM per label
    gmms = []
    for c in range(n_labels):
        Xc = pixels[labels.ravel() == c]

        if len(Xc) < max(n_gmm_components, 2):
            Xc = pixels[rng.choice(len(pixels), size=20, replace=True)]

        gmm_c = GaussianMixture(
            n_components=n_gmm_components,
            covariance_type="full",
            random_state=random_state,
            reg_covar=1e-6,
        )
        gmm_c.fit(Xc)
        gmms.append(gmm_c)

    energy_history = []
    prev_energy = None

    def neighbors_cost(labels, c):
        up = np.zeros_like(labels, dtype=bool)
        up[1:, :] = labels[:-1, :] != c

        down = np.zeros_like(labels, dtype=bool)
        down[:-1, :] = labels[1:, :] != c

        left = np.zeros_like(labels, dtype=bool)
        left[:, 1:] = labels[:, :-1] != c

        right = np.zeros_like(labels, dtype=bool)
        right[:, :-1] = labels[:, 1:] != c

        return (
            up.astype(float)
            + down.astype(float)
            + left.astype(float)
            + right.astype(float)
        )

    for em_it in range(max_em_iter):

        # --- E step (unary)
        unary = np.zeros((n_labels, H, W))
        for c in range(n_labels):
            ll = gmms[c].score_samples(pixels)
            unary[c] = (-ll).reshape(H, W)

        # --- ICM
        for _ in range(max_icm_iter):
            local_energy = np.zeros((n_labels, H, W))

            for c in range(n_labels):
                local_energy[c] = unary[c] + beta * neighbors_cost(labels, c)

            new_labels = np.argmin(local_energy, axis=0)
            if np.all(new_labels == labels):
                break
            labels = new_labels

        # --- M step
        for c in range(n_labels):
            Xc = pixels[labels.ravel() == c]

            if len(Xc) < max(n_gmm_components, 2):
                Xc = pixels[rng.choice(len(pixels), size=20, replace=True)]

            gmms[c] = GaussianMixture(
                n_components=n_gmm_components,
                covariance_type="full",
                random_state=random_state,
                reg_covar=1e-6,
            )
            gmms[c].fit(Xc)

        # --- Energy
        energy = 0.0
        for c in range(n_labels):
            mask = labels == c
            energy += unary[c][mask].sum()

        energy += beta * np.sum(labels[1:, :] != labels[:-1, :])
        energy += beta * np.sum(labels[:, 1:] != labels[:, :-1])

        energy_history.append(float(energy))

        if prev_energy is not None:
            rel = abs(prev_energy - energy) / max(abs(prev_energy), 1e-12)
            if rel < tol:
                break

        prev_energy = energy

    return labels, energy_history

# GMM-CRF

In [34]:
def gmm_crf_from_gmm(
    X,
    gmm,
    image_shape,
    beta=1.0,
    max_iter=10,
    tol=1e-3,
):
    """
    GMM-CRF segmentation using a pre-fitted GMM.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        Feature matrix.
    gmm : GaussianMixture
        Pre-fitted GMM model.
    image_shape : tuple (H, W)
        Shape of the image.
    beta : float
        Pairwise smoothness weight.
    max_iter : int
        Number of ICM iterations.
    tol : float
        Convergence tolerance.

    Returns
    -------
    labels : np.ndarray, shape (H, W)
    energy_history : list
    """

    H, W = image_shape
    n_labels = gmm.n_components

    # Unary: -log p(x | label)
    log_prob = gmm._estimate_weighted_log_prob(X)  # (N, K)
    unary = -log_prob.reshape(H, W, n_labels).transpose(2, 0, 1)

    # Init
    labels = np.argmin(unary, axis=0)

    energy_history = []
    prev_energy = None

    def pairwise_cost(labels, c):
        up = np.zeros_like(labels, dtype=bool)
        up[1:, :] = labels[:-1, :] != c

        down = np.zeros_like(labels, dtype=bool)
        down[:-1, :] = labels[1:, :] != c

        left = np.zeros_like(labels, dtype=bool)
        left[:, 1:] = labels[:, :-1] != c

        right = np.zeros_like(labels, dtype=bool)
        right[:, :-1] = labels[:, 1:] != c

        return (
            up.astype(float)
            + down.astype(float)
            + left.astype(float)
            + right.astype(float)
        )

    for it in range(max_iter):
        local_energy = np.zeros((n_labels, H, W))

        for c in range(n_labels):
            local_energy[c] = unary[c] + beta * pairwise_cost(labels, c)

        new_labels = np.argmin(local_energy, axis=0)
        labels = new_labels

        # Energy
        energy = 0.0
        for c in range(n_labels):
            mask = labels == c
            energy += unary[c][mask].sum()

        energy += beta * np.sum(labels[1:, :] != labels[:-1, :])
        energy += beta * np.sum(labels[:, 1:] != labels[:, :-1])

        energy_history.append(float(energy))

        if prev_energy is not None:
            rel = abs(prev_energy - energy) / max(abs(prev_energy), 1e-12)
            if rel < tol:
                break

        prev_energy = energy

    return labels, energy_history

# KB-SGMM

In [35]:
def gaussian_kernel(size, sigma=1.0):
    ax = np.linspace(-(size // 2), size // 2, size)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    return kernel / np.sum(kernel)


def kb_sgmm(dataset, gmm_model, lambda_=0.0, window_size=6):
    h, w = dataset.shape
    X_val = dataset.reshape(-1, 1)
    
    # GMM shape probabilities (h * w, n_classes)
    probs = gmm_model.predict_proba(X_val)
    probs_2d = probs.reshape(h, w, -1)

    # kernel = np.ones((window_size, window_size), dtype=np.float32) / (window_size ** 2)
    kernel = gaussian_kernel(window_size, sigma=window_size / 3)
    
    smoothed = np.stack([
        convolve(probs_2d[:, :, i], kernel, mode='constant')
        for i in range(probs_2d.shape[-1])
    ], axis=-1)

    corrected = lambda_ * probs_2d + (1 - lambda_) * smoothed

    corrected /= corrected.sum(axis=-1, keepdims=True)

    return np.argmax(corrected, axis=-1),corrected,probs_2d

In [36]:
def evaluate_clustering(true_labels, pred_labels):
    true_labels, pred_labels = true_labels.flatten(),pred_labels.flatten()
    return {
        "ARI": round(adjusted_rand_score(true_labels, pred_labels),2),
        "NMI": round(normalized_mutual_info_score(true_labels, pred_labels),2)
    }



In [37]:
def plot_clustering_results_list(d_name,results_list, h, w, methods=None, cmap='tab10'):
    """
    Plot clustering results from multiple datasets in a grid (1 row per dataset).

    Parameters:
    - results_list: list of dicts, each with keys as method names and values as label arrays
    - h, w: height and width to reshape the label arrays into images
    - methods: list of methods to include (optional)
    - cmap: colormap to use
    """

    n_datasets = len(results_list)

    # Determine all methods if not provided
    if methods is None:
        methods = list(results_list[0].keys())

    n_methods = len(methods)
    fig, axes = plt.subplots(nrows=n_datasets, ncols=n_methods, figsize=(4 * n_methods, 4 * n_datasets))

    # Ensure axes is 2D even if there's only one row or one column
    if n_datasets == 1:
        axes = np.expand_dims(axes, 0)
    if n_methods == 1:
        axes = np.expand_dims(axes, 1)

    for row_idx, results in enumerate(results_list):
        for col_idx, method in enumerate(methods):
            ax = axes[row_idx, col_idx]
            img = results[method].reshape(h, w)
            im = ax.imshow(img, cmap=cmap, vmin=0, vmax=np.max(img))
            ax.set_title(d_name+f"{row_idx+1} - {method}")
            ax.axis('off')

    plt.tight_layout()
    plt.savefig('clustering_'+d_name+'.png',dpi=300,bbox_inches='tight')
    plt.show()

In [38]:
def correct_clustering_labels(true_labels, predicted_labels):
    true_labels = np.asarray(true_labels)
    predicted_labels = np.asarray(predicted_labels)

    cm = confusion_matrix(true_labels, predicted_labels)

    # Hungarian algorithm to maximize correct label matching
    row_ind, col_ind = linear_sum_assignment(-cm)

    label_mapping = {col: row for row, col in zip(row_ind, col_ind)}
    
    corrected_labels = np.array([label_mapping[label] for label in predicted_labels])

    return corrected_labels



In [39]:
def run_clustering_pipeline(X, S, true_labels, n_clusters=3,
                            window_size=6, lambda_=0.3,
                            sgmm_max_iter=40, sgmm_tol=1e-6,
                            random_state=0):
    results = {}
    probas = {}
    h = w = int(np.sqrt(len(X)))
    dataset_reshaped = X.reshape(h, w)
    results['True_labels'] = true_labels.flatten()
    # --- KMEANS ---
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state).fit(X)
    results["KMeans"] = correct_clustering_labels(true_labels.flatten(),kmeans.labels_)

    # --- GMM classique ---
    gmm = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=random_state,tol=1e-5).fit(X)
    results["GMM"] =  correct_clustering_labels(true_labels.flatten(),gmm.predict(X))
    

    # --- Spatially Aided GMM ---
    sagmm = SpatiallyAidedGMM(n_clusters=n_clusters)
    sagmm.fit(X, S)
    results["SAGMM"] =  correct_clustering_labels(true_labels.flatten(),sagmm.predict(X, S))
    # --- GMM-CRF ---
    labels_crf, energy_crf = gmm_crf_from_gmm(
        X=X,
        gmm=gmm,
        image_shape=(h, w),
        beta=1.0,
        max_iter=10,
    )

    results["GMM-CRF"] = correct_clustering_labels(
    true_labels.flatten(),
    labels_crf.flatten()
    )

    # --- GMM-HMRF ---
    labels_hmrf, energy_hmrf = gmm_hmrf_from_gmm(
        X=X,
        gmm=gmm,
        image_shape=(h, w),
        beta=1.0,
        max_em_iter=10,
    )
    
    results["GMM-HMRF"] = correct_clustering_labels(
        true_labels.flatten(),
        labels_hmrf.flatten()
    )
    
    

    # --- SGMM ---
    labels_sgmm, mu_sgmm, sigma2_sgmm, probs_sgmm = sgmm(
        image=dataset_reshaped,
        K=n_clusters,
        window_size=window_size,
        max_iter=sgmm_max_iter,
        tol=sgmm_tol,
        random_state=random_state
    )
    results["SGMM"] =  correct_clustering_labels(true_labels.flatten(),labels_sgmm.flatten())
    # results["SGMM"] =  labels_sgmm.flatten()
    probas["SGMM"] = probs_sgmm

    
    # --- KB-SGMM (spatial smoothing) ---
    
    labels_kb, prob_kb, prob_gmm = kb_sgmm(dataset_reshaped, gmm, lambda_=lambda_, window_size=window_size)
    results["KB-SGMM"] =  correct_clustering_labels(true_labels.flatten(),labels_kb.flatten())
    # results["KB-SGMM"] =  labels_kb.flatten()
    probas["KB-SGMM"] = prob_kb.reshape(-1, n_clusters)
    probas["GMM"] = prob_gmm.reshape(-1, n_clusters)
   

    # --- Évaluation ---
    scores = {method: evaluate_clustering(true_labels, pred) for method, pred in results.items()}

    return results, scores, probas



In [40]:
X_indices = np.array([(i, j) for i in range(100) for j in range(100)])


In [ ]:
labels_sgmm, mu_sgmm, sigma2_sgmm, probs_sgmm = sgmm(
    image=sd1,
    K=5,
    window_size=window_size,
    max_iter=sgmm_max_iter,
    tol=sgmm_tol,
    random_state=42,
)

In [41]:

# --- Load datasets ---
with open("simple_datasets.pkl", "rb") as file:
    datasets = pickle.load(file)

In [42]:
# Exemple avec sd1
sd1 = datasets['SD1']['data']
label_sd1 = datasets['SD1']['labels']
X1 = sd1.reshape(-1, 1)
S1 = X_indices.reshape(-1, 2)  # coordonnées spatiales (si disponibles)
results1, scores1,proba1, = run_clustering_pipeline(X1, S1, label_sd1, n_clusters=5, window_size=7, lambda_=0.)

# Exemple avec sd2
sd2 = datasets['SD2']['data']
label_sd2 = datasets['SD2']['labels']
X2 = sd2.reshape(-1, 1)
results2, scores2, proba2 = run_clustering_pipeline(X2, S1, label_sd2, n_clusters=5,window_size=7, lambda_=0.)

In [43]:
scores1

{'True_labels': {'ARI': 1.0, 'NMI': 1.0},
 'KMeans': {'ARI': 0.53, 'NMI': 0.61},
 'GMM': {'ARI': 0.52, 'NMI': 0.6},
 'SAGMM': {'ARI': 0.71, 'NMI': 0.75},
 'GMM-CRF': {'ARI': 0.84, 'NMI': 0.85},
 'GMM-HMRF': {'ARI': 0.91, 'NMI': 0.9},
 'SGMM': {'ARI': 0.97, 'NMI': 0.96},
 'KB-SGMM': {'ARI': 0.94, 'NMI': 0.93}}

In [44]:
scores2

{'True_labels': {'ARI': 1.0, 'NMI': 1.0},
 'KMeans': {'ARI': 0.53, 'NMI': 0.6},
 'GMM': {'ARI': 0.55, 'NMI': 0.61},
 'SAGMM': {'ARI': 0.72, 'NMI': 0.75},
 'GMM-CRF': {'ARI': 0.86, 'NMI': 0.87},
 'GMM-HMRF': {'ARI': 0.9, 'NMI': 0.88},
 'SGMM': {'ARI': 0.81, 'NMI': 0.86},
 'KB-SGMM': {'ARI': 0.94, 'NMI': 0.93}}

In [45]:
# --- Load datasets ---
with open("geological_synthetic_datasets.pkl", "rb") as file:
    gsd = pickle.load(file)

In [46]:
X_gsd1 = gsd["GSD1"]["data"].reshape(-1, 1)
X_gsd2 = gsd["GSD2"]["data"].reshape(-1, 1)
X_gsd3 = gsd["GSD3"]["data"].reshape(-1, 1)
X_gsd4 = gsd["GSD4"]["data"].reshape(-1, 1)

labels_gsd1 = gsd["GSD1"]["labels"]
labels_gsd2 = gsd["GSD2"]["labels"]
labels_gsd3 = gsd["GSD3"]["labels"]
labels_gsd4 = gsd["GSD4"]["labels"]

gds_1, score_gds1, proba_gds1 = run_clustering_pipeline(
    X_gsd1, S1, labels_gsd1, n_clusters=5, window_size=7, lambda_=0.0
)



In [47]:
score_gds1

{'True_labels': {'ARI': 1.0, 'NMI': 1.0},
 'KMeans': {'ARI': 0.52, 'NMI': 0.59},
 'GMM': {'ARI': 0.52, 'NMI': 0.6},
 'SAGMM': {'ARI': 0.69, 'NMI': 0.74},
 'GMM-CRF': {'ARI': 0.83, 'NMI': 0.85},
 'GMM-HMRF': {'ARI': 0.9, 'NMI': 0.89},
 'SGMM': {'ARI': 0.81, 'NMI': 0.84},
 'KB-SGMM': {'ARI': 0.94, 'NMI': 0.93}}

In [48]:
gds_2, score_gds2, proba_gds2 = run_clustering_pipeline(
    X_gsd2, S1, labels_gsd2, n_clusters=5, window_size=7, lambda_=0.0
)


In [49]:
score_gds2

{'True_labels': {'ARI': 1.0, 'NMI': 1.0},
 'KMeans': {'ARI': 0.5, 'NMI': 0.6},
 'GMM': {'ARI': 0.5, 'NMI': 0.61},
 'SAGMM': {'ARI': 0.63, 'NMI': 0.72},
 'GMM-CRF': {'ARI': 0.75, 'NMI': 0.81},
 'GMM-HMRF': {'ARI': 0.87, 'NMI': 0.87},
 'SGMM': {'ARI': 0.94, 'NMI': 0.92},
 'KB-SGMM': {'ARI': 0.9, 'NMI': 0.9}}

In [50]:
gds_3, score_gds3, proba_gds3 = run_clustering_pipeline(
    X_gsd3, S1, labels_gsd3, n_clusters=5, window_size=7, lambda_=0.0
)

In [51]:
score_gds3

{'True_labels': {'ARI': 1.0, 'NMI': 1.0},
 'KMeans': {'ARI': 0.43, 'NMI': 0.56},
 'GMM': {'ARI': 0.47, 'NMI': 0.59},
 'SAGMM': {'ARI': 0.47, 'NMI': 0.61},
 'GMM-CRF': {'ARI': 0.68, 'NMI': 0.75},
 'GMM-HMRF': {'ARI': 0.75, 'NMI': 0.77},
 'SGMM': {'ARI': 0.74, 'NMI': 0.77},
 'KB-SGMM': {'ARI': 0.8, 'NMI': 0.8}}

In [52]:
gds_4, score_gds4, proba_gds4 = run_clustering_pipeline(
    X_gsd4, S1, labels_gsd4, n_clusters=5, window_size=7, lambda_=0.0
)

In [53]:
score_gds4

{'True_labels': {'ARI': 1.0, 'NMI': 1.0},
 'KMeans': {'ARI': 0.45, 'NMI': 0.56},
 'GMM': {'ARI': 0.46, 'NMI': 0.57},
 'SAGMM': {'ARI': 0.53, 'NMI': 0.63},
 'GMM-CRF': {'ARI': 0.72, 'NMI': 0.77},
 'GMM-HMRF': {'ARI': 0.69, 'NMI': 0.75},
 'SGMM': {'ARI': 0.8, 'NMI': 0.79},
 'KB-SGMM': {'ARI': 0.82, 'NMI': 0.8}}

In [54]:
all_experiments = {
    "SD1": {
        "results": results1,
        "scores": scores1,
        "probas": proba1,
    },
    "SD2": {
        "results": results2,
        "scores": scores2,
        "probas": proba2,
    },
    "GSD1": {
        "results": gds_1,
        "scores": score_gds1,
        "probas": proba_gds1,
    },
    "GSD2": {
        "results": gds_2,
        "scores": score_gds2,
        "probas": proba_gds2,
    },
    "GSD3": {
        "results": gds_3,
        "scores": score_gds3,
        "probas": proba_gds3,
    },
    "GSD4": {
        "results": gds_4,
        "scores": score_gds4,
        "probas": proba_gds4,
    },
}

In [55]:
with open("clustering_results.pkl", "wb") as f:
    pickle.dump(all_experiments, f)